# Rolling Normalized Real-World Scores

In [ ]:
from pathlib import Path
from typing import Any

from pfns.experiments.model_benchmarks.hashing import single_model_hash
from pfns.experiments.model_benchmarks.io import (
    REAL_WORLD_BUNDLE_KEYS,
    REAL_WORLD_REQUIRED_FILES,
    download_results_bundle_from_wandb,
    find_latest_real_world_bundle_for_model,
    load_dataframe_bundle,
    make_bundle_path,
    make_model_artifact_name,
    sanitize_wandb_artifact_component,
    save_dataframe_bundle,
)
from pfns.experiments.model_benchmarks.model_registry import get_models_from_names
from pfns.experiments.model_benchmarks.real_world_benchmarks import get_real_world_benchmark_dataset_count
from pfns.experiments.model_benchmarks.real_world_presets import get_real_world_preset
from pfns.experiments.model_benchmarks.workflows import (
    aggregate_real_world_results_from_bundles,
    alias_real_world_dataframe_bundle,
    build_real_world_run_metadata,
    real_world_bundle_is_compatible,
)
from pfns.run_evaluation_cli import compute_per_dataset_stats, summarize_results
from pfns.utils import build_exp_outputs_path, get_default_device

REAL_WORLD_PRESET = "tabarena"
SELECTED_MODELS = [
    "Linear_Attention_Non_Causal",
    "equal_params_new:DeltaNet_Comb_ST",
    # "equal_params_new:GLA_Comb_ST",
    # "equal_params_new:Gated_DeltaNet_Comb_ST",
    # "equal_params_new:Mamba2_Comb_ST",
    "equal_params:Transformer_Comb_ST",
]

preset_defaults = get_real_world_preset(REAL_WORLD_PRESET)
REAL_DATASET_EXPERIMENT: dict[str, Any] = preset_defaults["experiment"]
WANDB: dict[str, Any] = {
    "enabled": True,
    "artifact_name_real_eval": "real_eval_results",
    "entity": "icl_arch",
} | preset_defaults["wandb"]
OUTPUT_ROOT = build_exp_outputs_path(Path.cwd(), "real_world_eval")

models_to_compare = get_models_from_names(SELECTED_MODELS)
expected_real_metadata = build_real_world_run_metadata(
    experiment=REAL_DATASET_EXPERIMENT,
    device=get_default_device(),
)
cache_compatible_real_metadata = {
    key: value for key, value in expected_real_metadata.items() if key != "device"
}

print(f"Results are stored in: {OUTPUT_ROOT}")
print(f"Preset: {REAL_WORLD_PRESET}")
print("Cache compatibility ignores device metadata for portable W&B artifacts.")
print(f"Selected models: {list(models_to_compare)}")


In [ ]:
def _download_real_world_bundle_for_model(model_name: str, model_config: dict[str, Any]):
    model_hash = single_model_hash(
        model_name=model_name,
        model_config=model_config,
        experiment_payload=REAL_DATASET_EXPERIMENT,
    )
    artifact_name = make_model_artifact_name(
        base_artifact_name=WANDB["artifact_name_real_eval"],
        model_name=model_name,
        model_hash=model_hash,
    )
    downloaded_path = download_results_bundle_from_wandb(
        artifact_name=artifact_name,
        download_root=OUTPUT_ROOT / "wandb_model_cache" / "real_world",
        required_files=REAL_WORLD_REQUIRED_FILES,
        entity=WANDB["entity"],
        project=WANDB["artifact_project"],
    )
    if downloaded_path is None:
        return None, None

    downloaded_bundle = load_dataframe_bundle(
        downloaded_path,
        expected_keys=REAL_WORLD_BUNDLE_KEYS,
    )
    aliased_bundle, source_labels = alias_real_world_dataframe_bundle(
        downloaded_bundle,
        target_model_name=model_name,
    )
    if not real_world_bundle_is_compatible(
        aliased_bundle,
        model_name=model_name,
        expected_metadata=cache_compatible_real_metadata,
    ):
        bundle_metadata = aliased_bundle.get("metadata", {})
        mismatches = {
            key: {"downloaded": bundle_metadata.get(key), "expected": expected_value}
            for key, expected_value in cache_compatible_real_metadata.items()
            if bundle_metadata.get(key) != expected_value
        }
        print(f"Downloaded {model_name} from W&B but metadata was incompatible: {mismatches}")
        return None, None

    local_bundle_path = make_bundle_path(
        OUTPUT_ROOT / "real_world",
        f"{REAL_DATASET_EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}",
    )
    save_dataframe_bundle(
        dataframes=aliased_bundle["dataframes"],
        bundle_dir=local_bundle_path,
        experiment=REAL_DATASET_EXPERIMENT,
        run_metadata=expected_real_metadata,
    )
    if source_labels:
        print(
            f"Downloaded {model_name} from W&B labels {sorted(source_labels)}: "
            f"{downloaded_path}. Saved local alias bundle: {local_bundle_path}"
        )
    else:
        print(f"Downloaded {model_name} from W&B: {downloaded_path}")
    return local_bundle_path, aliased_bundle


bundles_by_model = {}
missing_models = []

for model_name, model_config in models_to_compare.items():
    bundle_path, bundle = find_latest_real_world_bundle_for_model(
        model_name,
        output_root=OUTPUT_ROOT,
        expected_metadata=cache_compatible_real_metadata,
    )
    if bundle is None and WANDB["enabled"]:
        bundle_path, bundle = _download_real_world_bundle_for_model(model_name, model_config)

    if bundle is None:
        missing_models.append(model_name)
        continue
    bundles_by_model[model_name] = {"path": bundle_path, "bundle": bundle}
    print(f"Loaded {model_name}: {bundle_path}")

if not bundles_by_model:
    raise RuntimeError("No compatible local or W&B-downloaded result bundles were found.")

all_results, results_by_model = aggregate_real_world_results_from_bundles(
    bundles_by_model,
    expected_splits=int(REAL_DATASET_EXPERIMENT["n_splits"]),
)
summary = summarize_results(all_results)
per_dataset = compute_per_dataset_stats(all_results)

if summary is None or per_dataset is None or per_dataset.empty:
    raise RuntimeError("Could not compute per-dataset results from loaded bundles.")

expected_dataset_count = get_real_world_benchmark_dataset_count(
    REAL_DATASET_EXPERIMENT["benchmark"]
)
print(f"Models loaded: {len(bundles_by_model)} / {len(models_to_compare)}")
print(f"Rows in all_results: {len(all_results)}")
dataset_count = per_dataset["dataset"].nunique()
print(f"Datasets covered: {dataset_count} / {expected_dataset_count}")
if missing_models:
    print("Missing compatible bundles for:", missing_models)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pfns.experiments.model_benchmarks.analysis import add_normalized_comparison_metrics
from pfns.experiments.model_benchmarks.constants import DEFAULT_COLORS
from pfns.experiments.model_benchmarks.plotting import (
    build_model_style_map,
    make_display_name_formatter,
    resolve_display_name_map,
)


def _find_repo_root(start_path=Path.cwd()):
    start_path = Path(start_path).resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / "PFNs" / "notebooks" / "real_world_experiments.ipynb").exists():
            return candidate
    return Path.cwd().resolve()


if "dataset_num_rows" not in all_results.columns:
    raise RuntimeError("all_results does not include dataset_num_rows.")

raw_metric_cols = ["accuracy", "roc_auc"]
dataset_metric_base = per_dataset.rename(
    columns={
        "dataset_num_rows_first": "dataset_num_rows",
        "accuracy_mean": "accuracy",
        "roc_auc_mean": "roc_auc",
    }
)[["model", "dataset", "dataset_num_rows", *raw_metric_cols]].dropna(
    subset=["dataset_num_rows", *raw_metric_cols]
)

normalized_scores = add_normalized_comparison_metrics(
    dataset_metric_base.melt(
        id_vars=["model", "dataset", "dataset_num_rows"],
        value_vars=raw_metric_cols,
        var_name="metric",
        value_name="value",
    ).assign(rep=lambda df: df["dataset"]),
    metric_keys=raw_metric_cols,
    higher_is_better_metrics={"accuracy", "roc_auc"},
    group_cols=("dataset",),
)

display_name_map = resolve_display_name_map(all_results)
display_name_map.update(
    {
        "Softmax_Transformer": "Softmax Non-Causal",
        "Transformer_Non_Causal": "Softmax Non-Causal",
        "equal_params:Transformer_Comb_ST": "Softmax Non-Causal",
        "Linear_Attention_Non_Causal": "Linear Non-Causal",
        "DeltaNet_Comb_ST": "DeltaNet",
        "DeltaNet_Comb_ST_Reference_New": "DeltaNet",
        "oracles:DeltaNet_Comb_ST": "DeltaNet",
    }
)
format_model_label = make_display_name_formatter(display_name_map=display_name_map)

model_order = [model for model in summary.index.tolist() if model in set(normalized_scores["model"])]
non_black_colors = [c for c in DEFAULT_COLORS if c.lower() not in {"#000000", "black", "k"}]
style_map = build_model_style_map(model_order, colors=non_black_colors)
style_map.update(
    {
        model_name: style
        for model_name, style in {
            "Softmax_Transformer": ("s", "-", "#08519C"),
            "Transformer_Non_Causal": ("s", "-", "#08519C"),
            "equal_params:Transformer_Comb_ST": ("s", "-", "#08519C"),
            "Linear_Attention_Non_Causal": ("<", "-", "#00796B"),
            "DeltaNet_Comb_ST": ("v", "-", "#B2182B"),
            "DeltaNet_Comb_ST_Reference_New": ("v", "-", "#B2182B"),
            "oracles:DeltaNet_Comb_ST": ("v", "-", "#B2182B"),
        }.items()
        if model_name in model_order
    }
)


In [ ]:
NORMALIZED_SCORE_ROLLING_WINDOW = 5
rolling_metric_specs = [
    ("normalized_roc_auc", "Normalized ROC-AUC ↑"),
    ("normalized_accuracy", "Normalized Accuracy ↑"),
]
rolling_score_frames = []

for (metric_key, model_name), model_df in normalized_scores[
    normalized_scores["metric"].isin({metric for metric, _ in rolling_metric_specs})
].groupby(["metric", "model"], observed=True):
    model_df = model_df[model_df["dataset_num_rows"].gt(0)].sort_values("dataset_num_rows").copy()
    if model_df.empty:
        continue
    rolling_score = model_df["value"].rolling(
        window=NORMALIZED_SCORE_ROLLING_WINDOW,
        center=True,
        min_periods=1,
    )
    model_df["score_roll_mean"] = rolling_score.mean()
    model_df["score_roll_std"] = rolling_score.std().fillna(0.0)
    rolling_score_frames.append(model_df)

if not rolling_score_frames:
    raise RuntimeError("No rolling-score data is available for the selected models.")

rolling_score_df = pd.concat(rolling_score_frames, ignore_index=True)

fig, axes = plt.subplots(
    1,
    len(rolling_metric_specs),
    figsize=(16, 6.2),
    dpi=400,
    sharey=True,
)
if len(rolling_metric_specs) == 1:
    axes = [axes]

for ax, (metric_key, metric_title) in zip(axes, rolling_metric_specs):
    metric_df = rolling_score_df[rolling_score_df["metric"] == metric_key]
    for model_name in model_order:
        model_df = metric_df[metric_df["model"] == model_name].sort_values("dataset_num_rows")
        if model_df.empty:
            continue
        marker, linestyle, color = style_map.get(model_name, ("o", "-", None))
        x = model_df["dataset_num_rows"].to_numpy(dtype=float)
        y = model_df["score_roll_mean"].to_numpy(dtype=float)
        y_std = model_df["score_roll_std"].to_numpy(dtype=float)
        ax.fill_between(
            x,
            np.clip(y - y_std, 0.0, 1.0),
            np.clip(y + y_std, 0.0, 1.0),
            color=color,
            alpha=0.12,
            linewidth=0,
            zorder=2,
        )
        ax.plot(
            x,
            y,
            marker=marker,
            linestyle=linestyle,
            color=color,
            linewidth=2.6,
            markersize=6.0,
            label=format_model_label(model_name),
            zorder=3,
        )
    ax.set_title(f"{metric_title} Rolling Mean")
    ax.set_xlabel("Dataset size (rows, log scale)")
    ax.set_xscale("log")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
axes[0].set_ylabel("Rolling normalized score")

fig.subplots_adjust(bottom=0.22, top=0.86, left=0.08, right=0.99, wspace=0.10)
handles, labels = axes[0].get_legend_handles_labels()
if handles:
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.03),
        ncol=min(max(1, len(handles)), 4),
        fontsize=11,
        frameon=True,
    )
fig.suptitle(
    f"Rolling Normalized Scores (window={NORMALIZED_SCORE_ROLLING_WINDOW} datasets)",
    y=0.98,
    fontsize=16,
)

rolling_pdf_path = _find_repo_root() / "PFNs" / "graphics" / "real_world_rolling_normalized_scores.pdf"
rolling_pdf_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(rolling_pdf_path, bbox_inches="tight", pad_inches=0.02)
print(f"Saved rolling normalized score figure to {rolling_pdf_path}")

plt.show()
